# 实验 1：认识 Qwen3-0.6B-Base 的整体结构

## 今天只回答一个问题

**一个句子进入 Qwen3-0.6B-Base 之后，会依次经过哪些主要模块？**

今天我们先不管每个模块里面具体怎么算，只看**模型的大体结构**。

我们会做三件事：

1. **先看一张总览图**，知道一个输入大致会经过哪些部分；
2. **再从真实的 Qwen3 模型中把结构列出来**，看看它实际包含哪些模块；
3. **最后把总览图和真实模型的结构放在一起对照**，确认我们看到的是同一个模型。

今天先不打开 Transformer Block 的内部。

也就是说，Attention、MLP、GQA、RoPE 和各种权重矩阵，今天都先不展开。这些内容会放到后面的实验里分别学习。

本实验会加载真实的 Qwen3-0.6B-Base 权重，模型文件大约 1.2GB，但我们**不会让模型生成文本，也不会训练模型**。

今天的任务很简单：

> **先知道 Qwen3 的“外面长什么样”。**


In [1]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

from qwen_kit import MODEL_PATH, show, show_path, observe, setup

show([('torch', torch.__version__),
      ('transformers', transformers.__version__),
      ('模型路径', show_path(MODEL_PATH))],
     header=['环境', '值'])

环境,值
torch,2.13.0+cu130
transformers,5.15.1
模型路径,models/Qwen3-0.6B-Base


In [2]:
# 只加载模型，不做生成；设备固定 CPU，dtype 用 bf16 —— 本实验只看形状，精度无关。
DEVICE = 'cpu'

# 全程关闭梯度。只研究推理，不训练。
torch.set_grad_enabled(False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
).to(DEVICE).eval()

config = model.config
N_LAYERS = config.num_hidden_layers
HIDDEN = config.hidden_size
VOCAB = config.vocab_size

show([('模型类', type(model).__name__, ''),
      ('device', str(next(model.parameters()).device), ''),
      ('dtype', str(next(model.parameters()).dtype), ''),
      ('已切到 eval（training=False）', not model.training, '✓ 表示确认通过'),
      ('已关闭梯度', not torch.is_grad_enabled(), ''),
      ('层数 / 宽度 / 词表', f'{N_LAYERS} / {HIDDEN} / {VOCAB}', '全部现推自 config'),
      ('参数量', f'{sum(p.numel() for p in model.parameters()):,}', '')],
     header=['加载结果', '值', '说明'])

加载结果,值,说明
模型类,Qwen3ForCausalLM,
device,cpu,
dtype,torch.bfloat16,
已切到 eval（training=False）,✓,✓ 表示确认通过
已关闭梯度,✓,
层数 / 宽度 / 词表,28 / 1024 / 151936,全部现推自 config
参数量,"596,049,920",


## 1. 先看模型的总览图

今天只观察模型的**外部结构**，先不打开 Transformer Block 的内部。

<div align="center">
  <img src="../assets/qwen3-backbone-overview.png" alt="Qwen3 顶层结构总览图" width="360" style="max-width: 100%; border: 1px solid #cbd5e1; border-radius: 8px;">
</div>

先把 Block 当作黑盒：它接收 hidden states，处理后交给下一阶段。Attention、MLP、GQA 和权重矩阵会在后续实验中分别学习。

### 总览图的代码版本

图上那五个阶段，每一步真实的输入和输出的形状是怎么样的？让模型计算一次就能看到。

In [3]:
PROMPT = '北京是中国的首都，巴黎是法国的'
input_ids = tokenizer(PROMPT, return_tensors='pt').input_ids.to(DEVICE)

setup(model, tokenizer)
qwen = observe(model, input_ids)

# 五个顶层阶段的形状，全部取自官方模型的 hook。
_skeleton = [
    ('input_ids', input_ids, 'B=1, S=9 的整数张量'),
    ('embed_tokens', qwen.embed, '查表：行号 → 1024 维向量'),
    ('layers.0', qwen.L[0].out, '保形'),
    ('layers.1', qwen.L[1].out, '保形  ⋯ 中间 26 层省略'),
    (f'layers.{N_LAYERS - 1}', qwen.L[N_LAYERS - 1].out, '保形'),
    ('final_norm', qwen.final_norm, 'RMSNorm，不改形状'),
    ('lm_head', qwen.logits, f'1024 → {VOCAB} 词表打分'),
]
show([(n, str(tuple(t.shape)), w) for n, t, w in _skeleton],
     header=['阶段', '输出形状', '说明'])

[transformers] `sdpa` attention does not support `output_attentions=True`. Please set your attention to `eager` if you want any of these features.


阶段,输出形状,说明
input_ids,"(1, 9)","B=1, S=9 的整数张量"
embed_tokens,"(1, 9, 1024)",查表：行号 → 1024 维向量
layers.0,"(1, 9, 1024)",保形
layers.1,"(1, 9, 1024)",保形 ⋯ 中间 26 层省略
layers.27,"(1, 9, 1024)",保形
final_norm,"(1, 9, 1024)",RMSNorm，不改形状
lm_head,"(1, 9, 151936)",1024 → 151936 词表打分


## 2. 从真实模型里找出它的主要部分

刚才我们已经把 Qwen3-0.6B-Base 加载好了。现在不再看手画的示意图，而是**直接看看眼前这个真实的 `model` 对象**。

这里我们**不把整个模型都打印出来**。因为 Qwen3 的内部结构比较大，而今天的目标只是先认识它最外面的几部分。所以，我们手动找出几个关键对象：


In [4]:
backbone = model.model
embedding = backbone.embed_tokens
blocks = backbone.layers
final_norm = backbone.norm
lm_head = model.lm_head

然后，让代码把这些对象的真实信息打印出来：

In [5]:
print(type(model).__name__)
print(f"├── model: {type(backbone).__name__}")
print(
    f"│   ├── embed_tokens: {type(embedding).__name__}"
    f"({embedding.num_embeddings:,}, {embedding.embedding_dim})"
)
print(
    f"│   ├── layers: {len(blocks)} × "
    f"{type(blocks[0]).__name__}"
)
print(f"│   └── norm: {type(final_norm).__name__}({config.hidden_size})")
print(
    f"└── lm_head: {type(lm_head).__name__}"
    f"({lm_head.in_features}, {lm_head.out_features}, bias={lm_head.bias is not None})"
)


Qwen3ForCausalLM
├── model: Qwen3Model
│   ├── embed_tokens: Embedding(151,936, 1024)
│   ├── layers: 28 × Qwen3DecoderLayer
│   └── norm: Qwen3RMSNorm(1024)
└── lm_head: Linear(1024, 151936, bias=False)


现在先不要急着记这些类名。我们只需要看懂这棵树：

* 最外层是 `Qwen3ForCausalLM`，也就是我们刚刚加载的整个模型；
* 里面的 `model` 是 `Qwen3Model`，负责模型主体的大部分计算；
* `embed_tokens` 把输入的 token 变成向量；
* `layers` 一共有 **28 个 `Qwen3DecoderLayer`**，它们会反复处理这些向量；
* `norm` 是主体计算结束后的最后一次归一化；
* 最后的 `lm_head` 再把结果变成对整个词表的分数。

到这里，我们暂时只关心一件事：

> **Qwen3 的主要部分分别在哪里？**

至于这 28 层里面到底做了什么，我们下一步再慢慢打开来看。


## 3. 小结：今天只记住这副外部骨架

```text
Input Token
    ↓
Embedding                model.model.embed_tokens
    ↓
28 × Transformer Block   model.model.layers
    ↓
Final RMSNorm            model.model.norm
    ↓
LM Head                  model.lm_head
    ↓
Output Logits
```

### 到这里，你应该已经能说清楚

先不用记住很多细节。只要能回答下面两个问题就够了：

1. **一个 token 序列进入 Qwen3-0.6B-Base 之后，大致会先后经过哪些部分？**

2. **`Qwen3ForCausalLM`、`Qwen3Model`、`embed_tokens`、`layers`、`norm` 和 `lm_head` 之间是什么关系？谁在里面，谁在外面？**

如果这两个问题你已经能够自己说出来，那么今天的目标就完成了。

本实验先到这里。

我们暂时不打开 `layers` 里面的内容。下一实验再选一个具体的部分，把它打开，看看它里面到底是怎么工作的。
